# Colon Cancer Tissue Classification using Vision Transformer (ViT)

**Dataset:** CRC-VAL-HE-7K from `owkin/nct-crc-he` (HuggingFace Hub)  
**Model:** `google/vit-base-patch16-224` (Vision Transformer pretrained on ImageNet-21k)  
**Platform:** Kaggle Notebook (GPU P100/T4)

## 9 Tissue Classes
| Code | Description |
|------|-------------|
| ADI | Adipose tissue |
| BACK | Background / glass |
| DEB | Debris |
| LYM | Lymphocytes |
| MUC | Mucus |
| MUS | Smooth muscle |
| NORM | Normal colon mucosa |
| STR | Cancer-associated stroma |
| TUM | Colorectal adenocarcinoma (Tumor) |

## Outputs
1. Training/Validation **Accuracy Graph**
2. Training/Validation **Loss Graph**
3. **Classification Report** (precision, recall, F1 per class)
4. **Confusion Matrix** heatmap
5. **Single-image inference**

> **Kaggle Setup:** Enable GPU → Settings → Accelerator → GPU T4 x2 or P100. Enable Internet access.

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────────────────────
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib seaborn

In [ ]:
# ── Cell 2: Imports & Constants ───────────────────────────────────────────────
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import evaluate
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

# ── Hyperparameters ──────────────────────────────────────────────────────────
MODEL_CHECKPOINT = "google/vit-base-patch16-224"
DATASET_NAME = "owkin/nct-crc-he"
NUM_EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_ACCUM_STEPS = 2  # effective batch size = 16 * 2 = 32
OUTPUT_DIR = "/kaggle/working/vit-colon-cancer"
SEED = 42

# ── Class Labels ─────────────────────────────────────────────────────────────
CLASS_NAMES = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]

LABEL_DESCRIPTIONS = {
    "ADI": "Adipose tissue",
    "BACK": "Background / glass",
    "DEB": "Debris",
    "LYM": "Lymphocytes",
    "MUC": "Mucus",
    "MUS": "Smooth muscle",
    "NORM": "Normal colon mucosa",
    "STR": "Cancer-associated stroma",
    "TUM": "Colorectal adenocarcinoma (Tumor)",
}

id2label = {i: name for i, name in enumerate(CLASS_NAMES)}
label2id = {name: i for i, name in enumerate(CLASS_NAMES)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Model:  {MODEL_CHECKPOINT}")
print(f"Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")

In [ ]:
# ── Cell 3: Load Dataset & Split ──────────────────────────────────────────────
dataset = load_dataset(DATASET_NAME)

# Use the 7K validation set (larger, balanced) — split 80/20 for train/val
full_ds = dataset["crc_val_he_7k"]
split = full_ds.train_test_split(test_size=0.2, stratify_by_column="label", seed=SEED)
train_ds = split["train"]
val_ds = split["test"]

print(f"Total images:      {len(full_ds)}")
print(f"Training set:      {len(train_ds)}")
print(f"Validation set:    {len(val_ds)}")
print(f"Number of classes:  {len(CLASS_NAMES)}")

# Show class distribution
print("\nClass distribution (train):")
labels_array = np.array(train_ds["label"])
for i, name in enumerate(CLASS_NAMES):
    count = (labels_array == i).sum()
    print(f"  {name:5s} ({LABEL_DESCRIPTIONS[name]:35s}): {count:5d}")

In [ ]:
# ── Cell 4: Preprocessing & Data Augmentation ────────────────────────────────
from torchvision.transforms import (
    Compose, RandomHorizontalFlip, RandomVerticalFlip,
    ColorJitter, RandomRotation, ToTensor, Normalize, Resize, CenterCrop
)

processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)
size = processor.size.get("shortest_edge", 224)
normalize = Normalize(mean=processor.image_mean, std=processor.image_std)

# Training: augmentation for histopathology (rotation-invariant)
train_transforms = Compose([
    Resize((size, size)),
    RandomHorizontalFlip(p=0.5),
    RandomVerticalFlip(p=0.5),
    RandomRotation(degrees=90),
    ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    ToTensor(),
    normalize,
])

# Validation: no augmentation
val_transforms = Compose([
    Resize((size, size)),
    CenterCrop(size),
    ToTensor(),
    normalize,
])

def preprocess_train(batch):
    batch["pixel_values"] = [train_transforms(img.convert("RGB")) for img in batch["image"]]
    return batch

def preprocess_val(batch):
    batch["pixel_values"] = [val_transforms(img.convert("RGB")) for img in batch["image"]]
    return batch

train_ds_transformed = train_ds.with_transform(preprocess_train)
val_ds_transformed = val_ds.with_transform(preprocess_val)

print(f"Image size: {size}x{size}")
print("Training augmentations: HFlip, VFlip, Rotation(90°), ColorJitter")
print("Preprocessing ready!")

In [ ]:
# ── Cell 5: Load ViT Model ────────────────────────────────────────────────────
model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(CLASS_NAMES),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_CHECKPOINT}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Output classes:       {model.config.num_labels}")
print(f"Labels: {model.config.id2label}")

In [ ]:
# ── Cell 6: Training Configuration ────────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy_metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_transformed,
    eval_dataset=val_ds_transformed,
    data_collator=DefaultDataCollator(),
    processing_class=processor,
    compute_metrics=compute_metrics,
)

print("Training configuration:")
print(f"  Epochs:              {NUM_EPOCHS}")
print(f"  Batch size:          {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"  Learning rate:       {LEARNING_RATE}")
print(f"  LR scheduler:        cosine with {WARMUP_RATIO:.0%} warmup")
print(f"  Weight decay:        {WEIGHT_DECAY}")
print(f"  FP16:                {training_args.fp16}")
print(f"  Output dir:          {OUTPUT_DIR}")

In [ ]:
# ── Cell 7: Train the Model ───────────────────────────────────────────────────
print("Starting training...")
train_result = trainer.train()

# Extract training history
log_history = trainer.state.log_history

train_losses = []
val_losses = []
val_accuracies = []
epochs = []

for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        val_losses.append(entry["eval_loss"])
        val_accuracies.append(entry.get("eval_accuracy", 0))
        epochs.append(entry.get("epoch", len(epochs) + 1))

print(f"\nTraining complete!")
print(f"Best validation accuracy: {max(val_accuracies):.4f}")
print(f"Final training loss:      {train_losses[-1]:.4f}")

In [ ]:
# ── Cell 8: Plot Accuracy & Loss Graphs ───────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss graph
ax1.plot(range(1, len(train_losses) + 1), train_losses, "b-o", label="Training Loss", linewidth=2)
ax1.plot(range(1, len(val_losses) + 1), val_losses, "r-o", label="Validation Loss", linewidth=2)
ax1.set_xlabel("Epoch", fontsize=12)
ax1.set_ylabel("Loss", fontsize=12)
ax1.set_title("Training & Validation Loss", fontsize=14, fontweight="bold")
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy graph
ax2.plot(range(1, len(val_accuracies) + 1), val_accuracies, "g-o", label="Validation Accuracy", linewidth=2)
ax2.set_xlabel("Epoch", fontsize=12)
ax2.set_ylabel("Accuracy", fontsize=12)
ax2.set_title("Validation Accuracy", fontsize=14, fontweight="bold")
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

# Annotate best accuracy
best_epoch = np.argmax(val_accuracies) + 1
best_acc = max(val_accuracies)
ax2.annotate(f"Best: {best_acc:.4f}\n(Epoch {best_epoch})",
             xy=(best_epoch, best_acc), xytext=(best_epoch + 0.5, best_acc - 0.05),
             arrowprops=dict(arrowstyle="->", color="darkgreen"),
             fontsize=10, color="darkgreen", fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR}/training_curves.png")

In [ ]:
# ── Cell 9: Classification Report & Confusion Matrix ─────────────────────────
predictions = trainer.predict(val_ds_transformed)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Classification Report
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)
report = classification_report(labels, preds, target_names=CLASS_NAMES, digits=4)
print(report)

# Confusion Matrix
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, linewidths=0.5, linecolor="gray")
ax.set_xlabel("Predicted Label", fontsize=13)
ax.set_ylabel("True Label", fontsize=13)
ax.set_title("Confusion Matrix — ViT on CRC-VAL-HE-7K", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

# Overall accuracy
overall_acc = (preds == labels).mean()
print(f"\nOverall Validation Accuracy: {overall_acc:.4f} ({overall_acc:.2%})")
print(f"Saved: {OUTPUT_DIR}/confusion_matrix.png")

In [ ]:
# ── Cell 10: Save Model ───────────────────────────────────────────────────────
save_path = os.path.join(OUTPUT_DIR, "final_model")
trainer.save_model(save_path)
processor.save_pretrained(save_path)

print(f"Model saved to: {save_path}")
print(f"Best validation accuracy: {max(val_accuracies):.4f}")

# Optional: Push to HuggingFace Hub (uncomment and set your username)
# from huggingface_hub import notebook_login
# notebook_login()
# HF_USERNAME = "your-username"
# trainer.push_to_hub(f"{HF_USERNAME}/vit-colon-cancer-tissue")
# print(f"Pushed to HuggingFace Hub: {HF_USERNAME}/vit-colon-cancer-tissue")

In [ ]:
# ── Cell 11: Single Image Inference ───────────────────────────────────────────
def classify_image(image_path):
    """Classify a single histopathology image into one of 9 tissue types."""
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)[0]

    results = []
    for i, (prob, name) in enumerate(zip(probs, CLASS_NAMES)):
        results.append({
            "label": name,
            "description": LABEL_DESCRIPTIONS[name],
            "confidence": f"{prob.item():.2%}",
            "score": prob.item(),
        })

    results.sort(key=lambda x: x["score"], reverse=True)

    print(f"\nClassification Results for: {image_path}")
    print("-" * 60)
    for r in results:
        bar = "█" * int(r["score"] * 30)
        print(f"  {r['label']:5s} ({r['description']:35s}) {r['confidence']:>7s} {bar}")
    print(f"\n  → Prediction: {results[0]['label']} ({results[0]['description']})")
    print(f"  → Confidence: {results[0]['confidence']}")
    is_cancer = results[0]["label"] == "TUM"
    print(f"  → Cancer detected: {'YES' if is_cancer else 'NO'}")
    return results

# Example usage (uncomment and set path to your image):
# results = classify_image("/kaggle/input/your-image.png")

# To test with a sample from the validation set:
sample = val_ds[0]
sample["image"].save("/tmp/test_sample.png")
classify_image("/tmp/test_sample.png")